# Day 072 — Exercise 5: AudioTranscriber Class

**What you'll build:** `AudioTranscriber` — a class that wraps the full Whisper pipeline with `transcribe`, `get_text`, `get_segments`, and `search` methods.

**Why it matters:** The class is the deliverable that slots into any app. One constructor call to configure, then call `.get_text` for plain transcription or `.search` for moment-finding.

In [ ]:
import os, tempfile

def _format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'

def format_transcript(result, include_timestamps=False):
    if not include_timestamps:
        return result.get('text', '').strip()
    lines = []
    for seg in result.get('segments', []):
        ts = _format_time(seg.get('start', 0.0))
        lines.append(f'[{ts}] {seg.get("text", "").strip()}')
    return '\n'.join(lines)

def extract_segments(result):
    out = []
    for seg in result.get('segments', []):
        logprob = seg.get('avg_logprob', -1.0)
        confidence = min(1.0, max(0.0, 1.0 + logprob))
        out.append({'start': float(seg.get('start', 0.0)),
                    'end':   float(seg.get('end', 0.0)),
                    'text':  seg.get('text', '').strip(),
                    'confidence': round(confidence, 4)})
    return out

def transcribe_audio(source, transcribe_fn=None, model='base'):
    if transcribe_fn is not None:
        return transcribe_fn(source)
    import whisper as _whisper
    mdl = _whisper.load_model(model)
    if isinstance(source, (bytes, bytearray)):
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
            f.write(source); tmp = f.name
        try:
            return mdl.transcribe(tmp)
        finally:
            os.unlink(tmp)
    return mdl.transcribe(str(source))

def search_transcript(result, query, case_sensitive=False):
    segments = extract_segments(result)
    if not case_sensitive:
        q = query.lower()
        return [s for s in segments if q in s['text'].lower()]
    return [s for s in segments if query in s['text']]

_MOCK_RESULT = {
    'text': ' Hello world. This is a test of speech recognition.',
    'language': 'en',
    'segments': [
        {'id': 0, 'start': 0.0, 'end': 3.2, 'text': ' Hello world.',
         'avg_logprob': -0.25, 'no_speech_prob': 0.01},
        {'id': 1, 'start': 3.2, 'end': 7.8,
         'text': ' This is a test of speech recognition.',
         'avg_logprob': -0.30, 'no_speech_prob': 0.02},
    ],
}
_mock_transcribe = lambda source: _MOCK_RESULT


## Task

Implement `AudioTranscriber`:

- `__init__(model='base', transcribe_fn=None)`: store both as instance attributes
- `transcribe(source) -> dict`: `transcribe_audio(source, transcribe_fn=self._transcribe_fn, model=self._model)`
- `get_text(source) -> str`: `format_transcript(self.transcribe(source))`
- `get_segments(source) -> list[dict]`: `extract_segments(self.transcribe(source))`
- `search(source, query, case_sensitive=False) -> list[dict]`: `search_transcript(self.transcribe(source), query, case_sensitive)`

## Your Implementation

In [ ]:
class AudioTranscriber:
    """Transcribe audio using openai-whisper with mock injection support."""

    def __init__(self, model: str = 'base',
                 transcribe_fn=None) -> None:
        raise NotImplementedError

    def transcribe(self, source) -> dict:
        """Transcribe audio. Returns full whisper result dict."""
        raise NotImplementedError

    def get_text(self, source) -> str:
        """Return plain transcript text (stripped)."""
        raise NotImplementedError

    def get_segments(self, source) -> list:
        """Return list of time-stamped segment dicts."""
        raise NotImplementedError

    def search(self, source, query: str,
               case_sensitive: bool = False) -> list:
        """Search for a query in the transcript segments."""
        raise NotImplementedError


In [ ]:
class AudioTranscriber:
    def __init__(self, model='base', transcribe_fn=None):
        self._model         = model
        self._transcribe_fn = transcribe_fn

    def transcribe(self, source):
        return transcribe_audio(source,
                                transcribe_fn=self._transcribe_fn,
                                model=self._model)

    def get_text(self, source):
        return format_transcript(self.transcribe(source))

    def get_segments(self, source):
        return extract_segments(self.transcribe(source))

    def search(self, source, query, case_sensitive=False):
        return search_transcript(self.transcribe(source), query,
                                 case_sensitive=case_sensitive)


## Automated checks

In [ ]:

score, total = 0, 5
try:
    tr = AudioTranscriber(transcribe_fn=_mock_transcribe)

    # transcribe returns the mock result dict
    result = tr.transcribe(b'audio bytes')
    assert isinstance(result, dict) and 'text' in result and 'segments' in result
    score += 1; print("✅ transcribe returns whisper result dict")

    # get_text returns stripped string
    text = tr.get_text(b'audio bytes')
    assert isinstance(text, str) and text == text.strip() and len(text) > 0
    score += 1; print("✅ get_text returns stripped transcript string")

    # get_segments returns list of dicts with required keys
    segs = tr.get_segments(b'audio bytes')
    assert isinstance(segs, list) and len(segs) == 2
    assert all('start' in s and 'confidence' in s for s in segs)
    score += 1; print("✅ get_segments returns list of segment dicts")

    # search finds matches
    hits = tr.search(b'audio bytes', 'hello')
    assert len(hits) == 1 and hits[0]['text'] == 'Hello world.'
    score += 1; print("✅ search finds matching segments")

    # transcribe_fn is stored and reused
    calls = [0]
    def _count(src):
        calls[0] += 1
        return _MOCK_RESULT
    tr2 = AudioTranscriber(transcribe_fn=_count)
    tr2.get_text(b'a'); tr2.get_segments(b'b')
    assert calls[0] == 2, f"Expected 2 calls, got {calls[0]}"
    score += 1; print("✅ transcribe_fn called once per method call")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
class AudioTranscriber:
    def __init__(self, model='base', transcribe_fn=None):
        self._model         = model
        self._transcribe_fn = transcribe_fn

    def transcribe(self, source):
        return transcribe_audio(source,
                                transcribe_fn=self._transcribe_fn,
                                model=self._model)

    def get_text(self, source):
        return format_transcript(self.transcribe(source))

    def get_segments(self, source):
        return extract_segments(self.transcribe(source))

    def search(self, source, query, case_sensitive=False):
        return search_transcript(self.transcribe(source), query,
                                 case_sensitive=case_sensitive)
```

**Why does each method call `self.transcribe` rather than caching the result?** In exercises, the source is always `b'fake audio'` — the mock is instant so re-transcribing is fine. In production, you would cache the result keyed by `hash(source)` to avoid loading the model twice for the same file.

</details>